# L02 · Anatomía de un sistema de ML en producción

**S2 · Lectura: Cap. 2 · 40–55 min**

## La pregunta

¿Qué puede fallar aunque el modelo responda?

## Qué harás

Reconocer las piezas de un sistema causal en producción.

- Lee la idea breve.
- Ejecuta la demostración y mira el gráfico.
- Completa las celdas marcadas.
- Escribe qué significa el resultado y qué no permite afirmar.

**Producto:** Un entorno local comprobado y un mapa simple del ciclo de vida.

> **Idea clave:** Un efecto necesita datos, contrato, evaluación, servicio y monitoreo.

La solución está en `la ruta de rescate docente (no incluida)`. Intenta primero el notebook del estudiante.

In [ ]:
# Localiza lab_common.py (carpeta actual o una superior)
import sys, pathlib
for _p in [pathlib.Path.cwd(), *list(pathlib.Path.cwd().parents)[:3]]:
    if (_p / "lab_common.py").exists():
        sys.path.insert(0, str(_p)); break

## 0. Preparación

In [ ]:
from lab_common import *
import numpy as np, pandas as pd, importlib, importlib.metadata as im, shutil, subprocess, sys, tempfile
from pathlib import Path
rng = setup()

In [ ]:
# El ciclo mínimo de un sistema causal
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 2.4))
steps = ["datos", "features", "efecto", "servicio", "monitoreo"]
ax.plot(range(len(steps)), [0] * len(steps), "o-", lw=3)
for i, step in enumerate(steps):
    ax.text(i, 0.08, step, ha="center", va="bottom", fontsize=11)
ax.set_xlim(-.35, len(steps)-.65); ax.set_ylim(-.2, .3); ax.axis("off")
ax.set_title("El modelo es solo una pieza del sistema")
plt.tight_layout(); plt.show()

## 1. Los cuatro requisitos, en versión causal (§2.3)
| Requisito | En un sistema predictivo | En un sistema **causal** |
|---|---|---|
| **Confiabilidad** | sigue funcionando pese a errores de datos o infraestructura | además: el efecto sigue siendo válido bajo los **supuestos de identificación declarados**, no solo que el servicio esté «arriba» |
| **Escalabilidad** | crece en datos/usuarios | además: poder re-estimar en subpoblaciones nuevas sin romper el pipeline de identificación |
| **Mantenibilidad** | otra persona puede entender y modificar el código | además: los **supuestos** están documentados y son auditables |
| **Adaptabilidad** | se ajusta a cambios del entorno | además: poder **detectar** cuándo un supuesto causal dejó de sostenerse |

> Estos cuatro, y no la precisión aislada del estimador, son el criterio con el que se evalúa cada notebook de la Serie B.

## 2. Las cinco señales de alerta (§2.6)
Si **dos o más** están presentes, el sistema no está listo para producción según el curso, por preciso que sea el estimador. Practiquemos con un sistema ficticio.

In [ ]:
SENALES = {
    "sin_versionado":      "No hay versión de datos ni de código asociadas al resultado",
    "skew_posible":        "La función que calcula covariables en entrenamiento y en producción no es la misma",
    "sin_supuestos":       "No existe ningún documento que declare los supuestos de identificación",
    "poblacion_opaca":     "Nadie más que quien entrenó puede decir en qué población vale el efecto",
    "sin_deteccion_drift": "Ningún mecanismo detecta si las covariables en producción cambian",
}
sistema_demo = {"sin_versionado": False, "skew_posible": True, "sin_supuestos": True,
                "poblacion_opaca": False, "sin_deteccion_drift": False}

In [ ]:
# ✏️ Completa tú — evaluar_sistema
# Pista: El libro: «si dos o más de estas señales están presentes…»
def evaluar_sistema(sistema):
    """Devuelve (n_senales_presentes, listo_para_produccion)."""
    raise NotImplementedError("cuenta las señales True; está listo solo si hay menos de 2")

In [ ]:
print(evaluar_sistema(sistema_demo))
lab_check("evaluar_sistema", lambda s: evaluar_sistema(s)[0],
          [((sistema_demo,), 2, 0), (({k: False for k in SENALES},), 0, 0)])
assert evaluar_sistema(sistema_demo)[1] is False and evaluar_sistema({k: False for k in SENALES})[1] is True

**Mira el resultado.** ¿Qué señales del sistema demo son las que activan la alerta, y cuál de los cuatro requisitos viola cada una?

**Respuesta:**

`skew_posible` (confiabilidad: el efecto servido no es el estimado) y `sin_supuestos` (mantenibilidad: nadie puede auditar el supuesto). Con dos señales, el sistema **no** está listo.



## 3. El esqueleto del proyecto y Git (§2.4, Lab 0)
Creamos el esqueleto en una carpeta **temporal** (no ensucia tu repositorio) y, si hay Git, lo inicializamos en local con un primer commit.

In [ ]:
ESQUELETO = ["data", "notebooks/serie_a_caso_estudio", "notebooks/serie_b_produccion",
             "src/identification", "src/estimation", "src/service", "tests", "scripts"]
tmp = Path(tempfile.mkdtemp(prefix="proyecto-causal-"))
for d in ESQUELETO:
    (tmp / d).mkdir(parents=True, exist_ok=True)
(tmp / "model_card.md").write_text("# Model card (plantilla)\n")
(tmp / "Dockerfile").write_text("# opcional: ver el Cap. 15\n")
(tmp / ".gitignore").write_text("data/\n__pycache__/\n")
print("proyecto-causal/")
for p in sorted(tmp.rglob("*")):
    rel = p.relative_to(tmp)
    if p.is_file() or not any(p.iterdir()):
        print("  ", rel)
git = shutil.which("git")
if git:
    def g(*a): return subprocess.run([git, *a], cwd=tmp, capture_output=True, text=True)
    g("init", "-q"); g("config", "user.email", "estudiante@local"); g("config", "user.name", "estudiante")
    g("add", "."); g("commit", "-q", "-m", "esqueleto del proyecto (Lab 0)")
    print("\nGit:", g("log", "--oneline").stdout.strip() or "sin commits")
else:
    print("\nGit no está instalado: instálalo para completar el Lab 0 (es obligatorio).")

## 4. Tu entorno, declarado y verificado
Un entorno que no está declarado no se puede reproducir (primera señal de alerta). Mira lo que hay instalado y **declara** qué necesita tu proyecto.

In [ ]:
def version(pkg):
    try: return im.version(pkg)
    except Exception: return None
tabla = pd.DataFrame({"paquete": ["numpy", "pandas", "scikit-learn", "scipy", "matplotlib", "networkx", "statsmodels",
                                   "econml", "shap", "fastapi", "httpx", "pydantic", "jsonschema"]})
tabla["version"] = tabla.paquete.map(lambda p: version(p) or "— (no instalado)")
print(f"Python {sys.version_info.major}.{sys.version_info.minor}   Git: {'sí' if shutil.which('git') else 'no'}   "
      f"Docker: {'sí' if shutil.which('docker') else 'no (opcional)'}")
show(tabla)

In [ ]:
# ✏️ Completa tú — ENV_DECLARATION
# Pista: Mira la tabla de arriba. Docker: ¿bloquea el trabajo si no lo tienes?
ENV_DECLARATION = {
    "python_min": None,       # p. ej. "3.11"
    "obligatorios": [],       # paquetes sin los cuales los labs no corren
    "opcionales": [],         # paquetes que enriquecen pero no bloquean
    "docker": None,           # "obligatorio" u "opcional"
}
raise NotImplementedError("rellena ENV_DECLARATION con lo que necesita TU proyecto")

In [ ]:
faltan = [p for p in ENV_DECLARATION["obligatorios"] if version(p) is None]
ok_py = tuple(map(int, ENV_DECLARATION["python_min"].split("."))) <= sys.version_info[:2]
print("Obligatorios que faltan:", faltan or "ninguno", "· Python suficiente:", ok_py)
assert ENV_DECLARATION["docker"] in ("opcional", "obligatorio") and len(ENV_DECLARATION["obligatorios"]) >= 3
assert not faltan, "instala los paquetes obligatorios antes de seguir"
print("✔ entorno declarado y verificado")

**Mira el resultado.** ¿Por qué conviene que Docker sea *opcional* en la declaración?

**Respuesta:**

Porque el objetivo del Lab 0 es poder **reproducir** el trabajo. Docker aporta aislamiento si lo tienes, pero no debe bloquear a quien no puede instalarlo; el requisito mínimo es un entorno Python declarado y un repositorio versionado.



## 5. Reentrenar **no** es inocuo (§2.5)
En un sistema predictivo, reentrenar con datos nuevos casi siempre mejora la métrica. En uno causal, los datos nuevos pueden **cambiar la población** para la que el efecto es representativo. Lo vemos: el efecto del tratamiento depende de una covariable $X$ ($\tau(X)=1+2X$); el tratamiento está aleatorizado.

In [ ]:
def ventana(n, media_x, rng):
    X = rng.normal(media_x, 1, n); T = rng.binomial(1, .5, n)
    Y = 2 + 1.5 * X + T * (1 + 2 * X) + rng.normal(0, 1, n)
    return X, T, Y
def estimar(X, T, Y):
    A = np.column_stack([np.ones(len(X)), T, X]); b = np.linalg.lstsq(A, Y, rcond=None)[0]
    r2 = 1 - ((Y - A @ b) ** 2).sum() / ((Y - Y.mean()) ** 2).sum()
    return b[1], r2
filas = []
for nombre, mx in [("ventana 1 (x̄ = 0)", 0.0), ("ventana 2, tras reentrenar (x̄ = 1)", 1.0)]:
    X, T, Y = ventana(4000, mx, rng); ate, r2 = estimar(X, T, Y)
    filas.append((nombre, round(r2, 3), round(ate, 3)))
show(pd.DataFrame(filas, columns=["datos usados", "R² (métrica predictiva)", "efecto estimado de T"]))

**Mira el resultado.** La métrica predictiva incluso mejora. ¿Qué pasó con el efecto y por qué importa?

**Respuesta:**

El **R² incluso sube y el efecto estimado pasó de ≈ 1 a ≈ 3**: la población cambió (otro x̄) y con ella el efecto promedio. Un monitoreo que solo mira la métrica predictiva no lo ve. Es el problema de validez externa que retomaremos en L24–L27.



## 6. Supuesto y primera fila de la bitácora
Esta es la fila de **arranque**: aún no hay estimando, pero ya declaramos qué vigilaremos.

In [ ]:
fila = bitacora(
    pregunta="¿Está mi entorno listo y declarado para estimar efectos de forma reproducible?",
    metodo="Checklist de 5 señales + declaración de entorno",
    estimando="ninguno (fila de arranque)",
    numero_ic="sin estimando; señales presentes en el sistema demo: %d" % evaluar_sistema(sistema_demo)[0],
    supuesto=supuesto("no observable", "declarado y acotado", "los supuestos de identificación se documentarán en cada lab"),
    decision="continuar", lab="L02")

---
## Qué queda para el libro
Ejercicios del capítulo cubiertos: ordenar los cuatro requisitos por facilidad de descuido (§2.7.1-3) y el Lab 0 (esqueleto + Git). Quedan: enumerar componentes según Huyen (§2.7.1-1) y explicar con tus palabras por qué reentrenar es delicado (§2.7.1-2). **Docker** es opcional en este curso (política P-01: todo local); si lo tienes, el Lab 0 del libro pide `docker run hello-world`.

*Material formativo (sin nota). Los datos son sintéticos: ver la ficha del escenario.*